# Post-Stage B Stage 4 Experiments — Google Colab Local-SSD + Safe Staging Runner

Prepared Stage 4 versioned notebook calling `src/bangla_smish/`.

### Mandatory Governance & Safety Gates
- **Canonical Notebook Immutability (Working Copy Required):** NEVER edit or execute `WORKSPACE_ROOT/experiments/colab/post_stage_b_stage4_experiments.ipynb` directly in Colab. First save a working copy OUTSIDE `DRIVE_WORKSPACE_REL` (`File -> Save a copy in Drive` into another folder) so Colab autosave never mutates the kit's verified notebook SHA256.
- **Expected First-Run Dependency Restart (Cell 4):** On a fresh Colab runtime, Cell 4 installs missing/bounded dependencies (preserving Colab CUDA PyTorch) and intentionally raises `RESTART REQUIRED`. Click `Runtime -> Restart session` in the SAME GPU runtime, then re-run from Cell 1.
- **Stage B synthetic generation is permanently stopped.**
- **T4 Benchmark (`RUN_MODE = "t4_benchmark"`)** runs ONLY the 5-unit calibration (`Exp D Fold 0 x 4 backbones + Exp B Fold 0 XLM-R`) on dev/fixture data (**zero held-out test/OOD exposure**) and requires explicit `AUTHORIZE_T4_BENCHMARK = True`.
- **Full Option A 160-Unit Grid (`RUN_MODE = "experiments_abcd"`)** remains strictly **`NO-GO`** until T4 benchmark gates pass and requires separate explicit `AUTHORIZE_FULL_TRAINING = True`.
- **Allowed-View Tokenizer Gate:** Cell 5 executes `run_allowed_view_tokenizer_collision_gate` on the 32 documented Stage 3 pairs (`raw`, `canon_masked`) at pinned HuggingFace revisions BEFORE Cell 6 can start, saving its preflight report to system temp and passing `TOK_GATE_REPORT` into Cell 6 for post-initialization export persistence.
- **Statistical Governance:** `CONFIRMATORY_DECLARATIONS_ENABLED = False` (`FAIL_OR_INCONCLUSIVE_CONFIRMATORY_DISABLED`); `confirmatory_mode=True` controls ONLY the Option A experimental grid and never enables superiority claims.
- **Checkpointing & Leases:** Local-SSD + UUID Drive staging + fail-closed single-writer leases requiring explicit `EXPECTED_STALE_WRITER_IDS` and `STALE_TAKEOVER_REASON` on authorized takeover.

In [ ]:
#@title 1. Stage 4 Runtime Configuration & Explicit Authorization Gates
RUN_MODE = 'gpu_smoke_test'  #@param ['gpu_smoke_test', 't4_benchmark', 'experiments_abcd']
AUTHORIZE_T4_BENCHMARK = False  #@param {type:'boolean'}
AUTHORIZE_FULL_TRAINING = False  #@param {type:'boolean'}
REQUIRE_GPU = True  #@param {type:'boolean'}
USE_GOOGLE_DRIVE = True  #@param {type:'boolean'}
DRIVE_WORKSPACE_REL = 'bangla_smish_post_stage_b_stage4_v5'  #@param {type:'string'}
ARTIFACT_RUN_ID = 'stage4-v5-smoke-01'  #@param {type:'string'}
RESUME = True  #@param {type:'boolean'}
ALLOW_STALE_WRITER_TAKEOVER = False  #@param {type:'boolean'}
EXPECTED_STALE_WRITER_IDS = ''  #@param {type:'string'}
EXPECTED_CORRUPT_LEASE_FILES = ''  #@param {type:'string'}
STALE_TAKEOVER_REASON = ''  #@param {type:'string'}

def validate_stage4_launch_settings():
    import re
    from pathlib import Path
    if RUN_MODE not in {'gpu_smoke_test', 't4_benchmark', 'experiments_abcd'}:
        raise ValueError(f'Unknown run mode {RUN_MODE!r}')
    rel_ws = DRIVE_WORKSPACE_REL.strip()
    if not rel_ws or Path(rel_ws).is_absolute() or '..' in Path(rel_ws).parts:
        raise ValueError('DRIVE_WORKSPACE_REL must be a non-empty relative path without parent traversal.')
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', ARTIFACT_RUN_ID):
        raise ValueError('Set ARTIFACT_RUN_ID using letters, digits, hyphens, or underscores only.')
    if RUN_MODE == 't4_benchmark' and not AUTHORIZE_T4_BENCHMARK:
        raise RuntimeError('5-unit T4 benchmark requires separate explicit owner authorization.')
    if RUN_MODE == 'experiments_abcd' and not AUTHORIZE_FULL_TRAINING:
        raise RuntimeError('Full 160-unit Option A training is NO-GO without explicit owner authorization.')
    if ALLOW_STALE_WRITER_TAKEOVER:
        if not STALE_TAKEOVER_REASON.strip():
            raise ValueError('Authorized stale-writer takeover requires a non-empty STALE_TAKEOVER_REASON.')
        if not EXPECTED_STALE_WRITER_IDS.strip() and not EXPECTED_CORRUPT_LEASE_FILES.strip():
            raise ValueError('Authorized stale-writer takeover requires explicit EXPECTED_STALE_WRITER_IDS or EXPECTED_CORRUPT_LEASE_FILES.')

# Reset prior preflight state so any Cell 1 parameter change requires re-running Cells 3 and 5.
PORTABLE_KIT_VERIFIED = False
VERIFIED_WORKSPACE_ROOT = None
VERIFIED_RUN_MODE = None
VERIFIED_ARTIFACT_RUN_ID = None
TOKENIZER_COLLISION_GATE_PASSED = False
TOK_GATE_REPORT = None
TOK_GATE_PREFLIGHT_PATH = None
validate_stage4_launch_settings()
print(f'Configured Stage 4 {RUN_MODE=} {ARTIFACT_RUN_ID=} {USE_GOOGLE_DRIVE=} {RESUME=} {ALLOW_STALE_WRITER_TAKEOVER=}')

In [ ]:
#@title 2. Verify Colab Runtime, GPU & Local SSD
import platform, shutil, subprocess, sys
from pathlib import Path
import torch

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
print('Python:', sys.version.split()[0], '| Platform:', platform.platform(), '| IN_COLAB:', IN_COLAB)
if shutil.which('nvidia-smi'):
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True).strip())
cuda_ok = torch.cuda.is_available()
print('PyTorch:', torch.__version__, '| CUDA available:', cuda_ok)
if REQUIRE_GPU and not cuda_ok:
    raise RuntimeError('CUDA GPU is required for Stage 4 benchmark/execution.')
if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| VRAM: {props.total_memory / (1024**3):.2f} GB')
ssd_path = '/content' if Path('/content').exists() else '.'
disk = shutil.disk_usage(ssd_path)
print(f'Free local SSD ({ssd_path}): {disk.free / (1024**3):.2f} GB')

In [ ]:
#@title 3. Mount Fresh Stage 4 Google Drive Kit & Resolve Paths
from pathlib import Path
import sys

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    WORKSPACE_ROOT = Path('/content/drive/MyDrive') / DRIVE_WORKSPACE_REL
else:
    WORKSPACE_ROOT = Path('/content') / DRIVE_WORKSPACE_REL

SRC_DIR = WORKSPACE_ROOT / 'src'
BUNDLE_DIR = WORKSPACE_ROOT / 'data' / 'processed' / 'post_stage_b_bundle_v2'
CONFIG_DIR = WORKSPACE_ROOT / 'experiments' / 'colab' / 'configs'
ARTIFACTS_DIR = WORKSPACE_ROOT / 'artifacts' / RUN_MODE / ARTIFACT_RUN_ID
CFG_MAP = {
    'gpu_smoke_test': 'gpu_smoke_test_config.json',
    't4_benchmark': 'post_stage_b_stage4_t4_benchmark_config.json',
    'experiments_abcd': 'post_stage_b_stage4_option_a_config.json',
}
CONFIG_PATH = CONFIG_DIR / CFG_MAP[RUN_MODE]
if not SRC_DIR.is_dir() or not BUNDLE_DIR.is_dir() or not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Missing Stage 4 kit files under {WORKSPACE_ROOT}. Stage kit via prepare_stage4_colab_kit.py first.')
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
#@title 4. Preflight, Install & Verify Compatible Dependencies (Preserving Colab CUDA PyTorch)
import json
from runpy import run_path

deps_mod = run_path(str(SRC_DIR / 'bangla_smish' / 'post_stage_b_colab_deps.py'))
cfg_models = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))['models']
dep_status = deps_mod['preflight_colab_dependencies'](
    cfg_models,
    install_missing=True,
    require_cuda=REQUIRE_GPU,
)
print('Verified Stage 4 dependency stack:', json.dumps(dep_status, indent=2))

In [ ]:
#@title 5. Verify Portable Kit, Frozen Masks, Config & 32-Pair Allowed-View Tokenizer Collision Gate
import json
import tempfile
from pathlib import Path
from bangla_smish.post_stage_b_data import sha256_file, verify_manifest
from bangla_smish.post_stage_b_integrity import verify_portable_colab_kit
from bangla_smish.post_stage_b_runner import collect_environment_info, load_config, seed_everything
from bangla_smish.post_stage_b_stats import CONFIRMATORY_DECLARATIONS_ENABLED, STATISTICAL_VALIDATION_STATUS
from bangla_smish.post_stage_b_tokenizer_gate import run_allowed_view_tokenizer_collision_gate

PORTABLE_KIT_VERIFIED = False
VERIFIED_WORKSPACE_ROOT = None
VERIFIED_RUN_MODE = None
VERIFIED_ARTIFACT_RUN_ID = None
TOKENIZER_COLLISION_GATE_PASSED = False
TOK_GATE_REPORT = None
TOK_GATE_PREFLIGHT_PATH = None
kit_summary = verify_portable_colab_kit(WORKSPACE_ROOT)
print('Verified portable kit:', json.dumps(kit_summary, indent=2))
manifest = verify_manifest(BUNDLE_DIR, verify_checksums=True)
print('Verified bundle:', manifest['manifest_version'], '| SHA256:', sha256_file(BUNDLE_DIR / 'dataset_manifest.json'))
cfg = load_config(CONFIG_PATH)
if cfg.run_mode != RUN_MODE:
    raise ValueError(f'Config run_mode {cfg.run_mode} != {RUN_MODE}')
seed_everything(cfg.seed)
if cfg.tokenizer_revisions:
    TOK_GATE_PREFLIGHT_PATH = (
        Path(tempfile.gettempdir())
        / f'stage4_tokenizer_collision_gate_{RUN_MODE}_{ARTIFACT_RUN_ID}_{cfg.config_hash()[:12]}.json'
    )
    TOK_GATE_REPORT = run_allowed_view_tokenizer_collision_gate(
        BUNDLE_DIR,
        cfg,
        allowed_views=tuple(v for v in cfg.text_views if v in {'raw', 'canon_masked'}),
        export_path=TOK_GATE_PREFLIGHT_PATH,
    )
    TOKENIZER_COLLISION_GATE_PASSED = bool(TOK_GATE_REPORT['passed'])
    print(
        'Tokenizer collision gate PASSED:',
        TOK_GATE_REPORT['total_checks'],
        'checks, collisions=',
        TOK_GATE_REPORT['collision_count'],
        '| preflight_report=',
        TOK_GATE_PREFLIGHT_PATH,
    )
elif RUN_MODE != 'gpu_smoke_test':
    raise RuntimeError('Fail closed: tokenizer_revisions must be pinned for benchmark/confirmatory runs.')
PORTABLE_KIT_VERIFIED = True
VERIFIED_WORKSPACE_ROOT = WORKSPACE_ROOT
VERIFIED_RUN_MODE = RUN_MODE
VERIFIED_ARTIFACT_RUN_ID = ARTIFACT_RUN_ID
print('Statistical governance:', STATISTICAL_VALIDATION_STATUS, '| confirmatory_declarations_enabled=', CONFIRMATORY_DECLARATIONS_ENABLED)
print('Loaded config:', cfg.run_name, '| config_hash:', cfg.config_hash())
print('Environment:', json.dumps(collect_environment_info(), indent=2))

In [ ]:
#@title 6. Execute Stage 4 Pipeline (Local-SSD + Immutable Drive Staging + Safe Leases)
from bangla_smish.post_stage_b_runner import run_post_stage_b_pipeline

validate_stage4_launch_settings()
_required_globals = ('WORKSPACE_ROOT', 'BUNDLE_DIR', 'ARTIFACTS_DIR', 'CONFIG_PATH', 'CFG_MAP', 'cfg')
_missing_globals = [name for name in _required_globals if name not in globals()]
if _missing_globals:
    raise RuntimeError(f'Missing prerequisite cell state {_missing_globals}; run Cells 1 through 5 in order before Cell 6.')
if (
    not globals().get('PORTABLE_KIT_VERIFIED', False)
    or globals().get('VERIFIED_WORKSPACE_ROOT') != WORKSPACE_ROOT
    or globals().get('VERIFIED_RUN_MODE') != RUN_MODE
    or globals().get('VERIFIED_ARTIFACT_RUN_ID') != ARTIFACT_RUN_ID
    or ARTIFACTS_DIR != WORKSPACE_ROOT / 'artifacts' / RUN_MODE / ARTIFACT_RUN_ID
    or CONFIG_PATH != WORKSPACE_ROOT / 'experiments' / 'colab' / 'configs' / CFG_MAP[RUN_MODE]
    or cfg.run_mode != RUN_MODE
):
    raise RuntimeError('Stale or unverified preflight state detected; re-run Cells 3 and 5 after any Cell 1 change.')
if cfg.tokenizer_revisions and (
    not globals().get('TOKENIZER_COLLISION_GATE_PASSED', False)
    or not isinstance(globals().get('TOK_GATE_REPORT'), dict)
):
    raise RuntimeError('Pre-training 32-pair allowed-view tokenizer collision gate has not passed.')
stale_ids = [s.strip() for s in EXPECTED_STALE_WRITER_IDS.split(',') if s.strip()] or None
corrupt_files = [s.strip() for s in EXPECTED_CORRUPT_LEASE_FILES.split(',') if s.strip()] or None
summary = run_post_stage_b_pipeline(
    bundle_dir=BUNDLE_DIR,
    artifacts_dir=ARTIFACTS_DIR,
    cfg=cfg,
    resume=RESUME,
    verify_checksums=True,
    allow_stale_takeover=ALLOW_STALE_WRITER_TAKEOVER,
    expected_stale_writer_ids=stale_ids,
    expected_corrupt_files=corrupt_files,
    takeover_reason=STALE_TAKEOVER_REASON or None,
    tokenizer_gate_report=globals().get('TOK_GATE_REPORT'),
)
print('Completed units total:', summary['completed_unit_count'], '/', summary['expected_unit_count'])

In [ ]:
#@title 7. Inspect Exported Telemetry, Optimizer Steps, Retraining Metadata & Results
import json, pandas as pd
exports_dir = ARTIFACTS_DIR / 'exports'
if not exports_dir.is_dir():
    print('Exports directory not found yet:', exports_dir)
else:
    print('Exported files:', [p.name for p in sorted(exports_dir.glob('*'))])
    tok_gate_path = exports_dir / 'tokenizer_collision_gate.json'
    if tok_gate_path.exists():
        tgate = json.loads(tok_gate_path.read_text(encoding='utf-8'))
        print('Tokenizer gate export:', 'PASSED' if tgate.get('passed') else 'FAILED', '| checks=', tgate.get('total_checks'), '| collisions=', tgate.get('collision_count'))
    retrain_path = exports_dir / 'retraining_metadata.json'
    if retrain_path.exists():
        rmeta = json.loads(retrain_path.read_text(encoding='utf-8'))
        print('Retraining metadata:', rmeta['unit_count'], 'units | status=', rmeta['statistical_validation_status'])
    csv_path = exports_dir / 'aggregated_results.csv'
    if csv_path.exists() and csv_path.stat().st_size > 0:
        print(pd.read_csv(csv_path).to_string(index=False))